# EmbeddingGemma 2 — 日本語＋英語を同時認識する Reader 学習（Gemma4Audio eager修正版）

共通の音声 Encoder と**日英共通の CTC Reader + 1つの語彙テーブル**を作ります。
言語ごとにファイルやパラメーターを切り替えず、`./cats -a eg2-native-asr -L auto` で認識する方式です。

デモとして FLEURS の日本語 `ja_jp` と英語 `en_us` を混合して学習しますが、
**各発話が単一言語のFLEURSだけでは、1発話内の日英切り替え性能は保証されません。**
日本語と英語が実際に混在する音声＋正解転写も追加して学習してください。

例：`今日はOpenAIのモデルをtestします` / `I use 日本語 recognition today`。
文字変換では英語の単語境界 `▁` を保持し、日本語には自動的な空白を挿入しません。

モデル重みは同梱しません。実用精度（日本語CER／英語WER／混合CER）は学習後に評価が必要です。
GPU ランタイムを推奨します。Google / Hugging Face / FLEURS の利用条件を確認してください。

T4 (16GB VRAM) を含む NVIDIA GPU に対応します。BF16 非対応なら FP32 を選びます。CPU も実行可能ですが低速です。

In [ ]:
# @title ① 設定（英日混合）
VOCAB_MODE = 'word'  # @param ['word','character']
MAX_VOCAB = 6000  # @param {type:'integer'}
TRAIN_LIMIT_PER_LANGUAGE = 500  # @param {type:'integer'}
DEV_LIMIT_PER_LANGUAGE = 70  # @param {type:'integer'}
EPOCHS = 5  # @param {type:'integer'}
BATCH_SIZE = 2  # @param {type:'integer'}
DEVICE_PREFERENCE = 'auto'  # @param ['auto','cuda','cpu']
WORD_TABLE_BATCH = 16  # @param {type:'integer'}
EMBEDDING_DTYPE = 'auto'  # @param ['auto','fp32','bf16']
INITIALIZE_FROM_ENGLISH = True  # @param {type:'boolean'}
USE_FLEURS = True  # @param {type:'boolean'}
# USE_FLEURS=False: UTF-8 JSONL {"id":...,"audio":...,"text":...}
# 日本語・英語・日英混合の音声を含める。30秒以下の発話に分割。
TRAIN_MANIFEST = '/content/mixed_train.jsonl'  # @param {type:'string'}
DEV_MANIFEST = '/content/mixed_dev.jsonl'  # @param {type:'string'}
SAVE_TO_DRIVE = False  # @param {type:'boolean'}
BROWSER_DOWNLOAD = True  # @param {type:'boolean'}

from pathlib import Path
BASE=Path('/content/eg2_mix')
TOOLS=BASE/'tools'; MODEL=BASE/'source_encoder'; DATA=BASE/'dataset'
VOCAB=BASE/'reader_vocab.txt'; TABLE=BASE/'word_table.npy'
CACHE_TRAIN=BASE/'cache/train'; CACHE_DEV=BASE/'cache/dev'
TRAINED=BASE/'trained'; READY=BASE/'models-mix'; OUTPUT=BASE/'embeddinggemma2-reader-en-ja.zip'
assert VOCAB_MODE in ('word','character') and MAX_VOCAB>0
import torch
if DEVICE_PREFERENCE not in ('auto','cuda','cpu'):
    raise ValueError('DEVICE_PREFERENCE: auto / cuda / cpu を選択')
DEVICE = ('cuda' if torch.cuda.is_available() else 'cpu') if DEVICE_PREFERENCE == 'auto' else DEVICE_PREFERENCE
if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('CUDA がありません。Colab のランタイム > ランタイムのタイプを変更 > T4 GPU 等を選択してください')
if WORD_TABLE_BATCH < 1:
    raise ValueError('WORD_TABLE_BATCH は1以上にしてください')
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    bf16 = torch.cuda.is_bf16_supported()
    print(f'GPU: {props.name} / {props.total_memory / 2**30:.1f} GiB / compute {props.major}.{props.minor} / BF16={bf16}')
    if EMBEDDING_DTYPE == 'bf16' and not bf16:
        raise ValueError('このGPUは BF16 非対応です。EMBEDDING_DTYPE=auto または fp32 を選択してください')
    print('EmbeddingGemma 2 演算精度:', 'BF16' if (EMBEDDING_DTYPE == 'bf16' or EMBEDDING_DTYPE == 'auto' and bf16) else 'FP32')
else:
    if EMBEDDING_DTYPE == 'bf16':
        raise ValueError('CPU 実行は EMBEDDING_DTYPE=auto または fp32 を選択してください')
    print('CPU モード: FP32 (遅くなります)')
print('選択デバイス:', DEVICE)

In [ ]:
# @title ② Python依存をインストール（実行時のみ）
import sys,subprocess
subprocess.run([sys.executable,'-m','pip','install','-q',
                'transformers>=5.19.0','sentence-transformers>=6.1.0',
                'huggingface_hub','datasets','soundfile','scipy',
                'sudachipy','sudachidict_core','torchcodec'],check=True)
for directory in (TOOLS,MODEL,DATA,CACHE_TRAIN,CACHE_DEV,TRAINED,READY):directory.mkdir(parents=True,exist_ok=True)
print('依存ライブラリ準備完了')

In [ ]:
# @title ③ 日本語学習・変換ツールを展開（別のソースアップロード不要）
import base64,io,zipfile
BUNDLE = """..."""
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))) as z:
    expected={'eg2_ja_vocab.py','eg2_mix_vocab.py','eg2_ja_table.py','eg2_ja_cache.py','eg2_ja_train.py','eg2_ja_fleurs.py','convert_eg2_reader.py','convert_eg2_table.py','eg2_bundle.py'}
    assert set(z.namelist())==expected
    z.extractall(TOOLS)
print('ツール:',len(expected))

## データを準備

`USE_FLEURS=True` なら FLEURS 日本語・英語をそれぞれ取得し、一つの学習リストに結合します。
同じ発話内の日本語＋英語にも対応させるには `USE_FLEURS=False` として、自分で録音した混合音声＋転写データを
`TRAIN_MANIFEST` と `DEV_MANIFEST` に指定してください。
必ず話者の違う検証データも用意してください（音声の学習・検証への重複投入は禁止）。

In [ ]:
# @title ④ 日本語・英語データをそれぞれ取得して統合
import json
if USE_FLEURS:
    for language in ('ja_jp','en_us'):
        subprocess.run([sys.executable,'-u',str(TOOLS/'eg2_ja_fleurs.py'),
            '--out',str(DATA/language),'--language',language,
            '--train-limit',str(TRAIN_LIMIT_PER_LANGUAGE),
            '--dev-limit',str(DEV_LIMIT_PER_LANGUAGE)],check=True)
    # IDs must stay unique: the audio cache uses IDs as file names.
    for split in ('train','dev'):
        target=DATA/(split+'.jsonl')
        with target.open('w',encoding='utf8') as out:
            for language in ('ja_jp','en_us'):
                for row in (DATA/language/(split+'.jsonl')).read_text(encoding='utf8').splitlines():
                    item=json.loads(row);item['id']=language+'_'+item['id']
                    out.write(json.dumps(item,ensure_ascii=False)+'\n')
    TRAIN_MANIFEST=str(DATA/'train.jsonl'); DEV_MANIFEST=str(DATA/'dev.jsonl')
assert Path(TRAIN_MANIFEST).is_file() and Path(DEV_MANIFEST).is_file()
print('学習:',TRAIN_MANIFEST,'検証:',DEV_MANIFEST)

必要なら Hugging Face の利用条件に同意し、認証を行ってください。下のセルでは `huggingface_hub.snapshot_download` を使い、**Encoder およびテキスト埋め込み用の公式重み**を取得します。Google のモデル利用規約・アクセス条件が適用されます。

In [ ]:
# @title 任意：Hugging Face 認証（アクセスが拒否された場合だけ、このセルを手動で実行）
# from huggingface_hub import login
# login()
print('認証不要ならこのまま次へ進みます。')

In [ ]:
# @title ⑤ EmbeddingGemma 2 公式モデルを取得
from huggingface_hub import snapshot_download,hf_hub_download
snapshot_download(repo_id='google/embeddinggemma-2',local_dir=str(MODEL),
    allow_patterns=['*.json','*.safetensors','*.model','tokenizer*','merges.txt','vocab.*'])
print('encoder / embedding model:',MODEL)

In [ ]:
# @title ⑥ 英日共通語彙を生成（▁は英語の単語境界）
subprocess.run([sys.executable,str(TOOLS/'eg2_mix_vocab.py'),
    '--train',str(TRAIN_MANIFEST),'--mode',VOCAB_MODE,
    '--max-vocab',str(MAX_VOCAB),'--out',str(VOCAB)],check=True)
print('共通語彙:',len(VOCAB.read_text(encoding='utf8').splitlines()))

In [ ]:
# @title ⑦ 日英共通単語の768次元ベクトルを生成（実際のEmbeddingGemma 2を使用）
subprocess.run([sys.executable,'-u',str(TOOLS/'eg2_ja_table.py'),
    '--model-dir',str(MODEL),'--vocab',str(VOCAB),'--out',str(TABLE),
    '--device',DEVICE,'--dtype',EMBEDDING_DTYPE,'--batch',str(WORD_TABLE_BATCH)],check=True)
print('word_table.npy 完成:',TABLE)

In [ ]:
# @title ⑧ 日英の音声 → 512次元 Soft Tokens のキャッシュ（詳細ログ付き）
# 失敗時は最後のCalledProcessErrorだけでなく、内部Python tracebackを画面とログに保存。
# 既にできた .npy は再利用するので、失敗後の再実行で最初から計算し直しません。
import subprocess,sys,torch

# Select the actual device *now*. The runtime/GPU may have changed since ①.
cuda_now = bool(torch.cuda.is_available())
if DEVICE_PREFERENCE == 'auto':
    DEVICE = 'cuda' if cuda_now else 'cpu'
if DEVICE == 'cuda' and not cuda_now:
    raise RuntimeError('CUDAが利用できません。ColabのランタイムをT4 GPUに切り替えて①と⑧を実行してください。')
if DEVICE == 'cuda':
    print('⑧ CUDA GPU:',torch.cuda.get_device_name(0), ' dtype:',EMBEDDING_DTYPE)
else:
    print('⑧ CPUで実行中 (T4を使う場合: ランタイム→ランタイムのタイプを変更→T4 GPU)。',flush=True)

def run_cache_logged(split, manifest, out_dir):
    logfile=BASE/('eg2_cache_'+split+'.log')
    command=[sys.executable,'-u',str(TOOLS/'eg2_ja_cache.py'),
        '--manifest',str(manifest),'--model-dir',str(MODEL),
        '--out',str(out_dir),'--device',DEVICE,'--dtype',EMBEDDING_DTYPE]
    print('>>>',split,manifest,' | log:',logfile,flush=True)
    with logfile.open('w',encoding='utf-8') as log:
        with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,encoding='utf-8',errors='replace',bufsize=1) as proc:
            for line in proc.stdout:
                print(line,end='',flush=True)
                log.write(line)
            returncode=proc.wait()
    if returncode:
        print('失敗時の詳細ログ:',logfile)
        print('Colabで確認: !tail -n 100 '+str(logfile))
        raise RuntimeError(f'{split} 音声キャッシュ失敗 (code={returncode})。上のtracebackを参照してください。')

for split,manifest,out_dir in [('train',TRAIN_MANIFEST,CACHE_TRAIN),('dev',DEV_MANIFEST,CACHE_DEV)]:
    run_cache_logged(split,manifest,out_dir)
print('音声特徴のキャッシュを生成しました')

In [ ]:
# @title ⑨ 既存の英語 Reader v2 を初期値として取得（日英共通Readerに追加学習する）
INIT=[]
if INITIALIZE_FROM_ENGLISH:
    pretrained=hf_hub_download(repo_id='mlboydaisuke/EmbeddingGemma-2-Transcribe-Reader',filename='reader.pt')
    INIT=['--init-reader',str(pretrained)]
    print('英語Readerを初期値にします。日本語辞書のみの差し替えではありません。')
else:
    print('ランダム初期化から学習します。より多くの音声・計算が必要です。')

In [ ]:
# @title ⑩ 日英共通 Reader を CTC 追加学習（例外とGPUメモリを詳細ログに保存）
import subprocess,sys,torch

if not (CACHE_TRAIN/'index.jsonl').is_file():
    raise FileNotFoundError('⑧のtrainキャッシュがありません: '+str(CACHE_TRAIN/'index.jsonl'))
if not (CACHE_DEV/'index.jsonl').is_file():
    raise FileNotFoundError('⑧のdevキャッシュがありません: '+str(CACHE_DEV/'index.jsonl'))
if not VOCAB.is_file() or not TABLE.is_file():
    raise FileNotFoundError('⑥の語彙または⑦の単語テーブルがありません')
# ①の実行後にGPUが切り替わった場合でも、今のランタイムから再検出。
if DEVICE_PREFERENCE == 'auto':
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE.startswith('cuda') and not torch.cuda.is_available():
    raise RuntimeError('CUDAが見つかりません。ランタイム→ランタイムのタイプを変更→T4 GPU を選択してください')
print('⑩ device =',DEVICE)
if DEVICE.startswith('cuda'):
    print('⑩ GPU =',torch.cuda.get_device_name(0))

LOGFILE=BASE/'eg2_train_ctc.log'

def run_reader_train(batch_size):
    cmd=[sys.executable,'-u',str(TOOLS/'eg2_ja_train.py'),
        '--train',str(CACHE_TRAIN/'index.jsonl'), '--dev',str(CACHE_DEV/'index.jsonl'),
        '--vocab',str(VOCAB), '--table',str(TABLE), '--out',str(TRAINED),
        '--mixed', '--layers','6','--epochs',str(EPOCHS),'--batch',str(batch_size),
        '--max-eval','30','--device',DEVICE,*INIT]
    print(f'CTC train | batch={batch_size} | log={LOGFILE}',flush=True)
    with LOGFILE.open('a',encoding='utf-8') as log:
        log.write('\n=== batch '+str(batch_size)+' ===\n');log.flush()
        with subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,encoding='utf-8',errors='replace',bufsize=1) as proc:
            last=[]
            for line in proc.stdout:
                print(line,end='',flush=True)
                log.write(line); log.flush()
                last.append(line)
                if len(last)>120:last.pop(0)
            code=proc.wait()
    return code, ''.join(last)

status,output=run_reader_train(BATCH_SIZE)
if status and BATCH_SIZE>1 and ('out of memory' in output.lower() or 'cuda error: out of memory' in output.lower()):
    print('T4 CUDA OOMを検出。batch=1で一度だけ学習し直します。',flush=True)
    if torch.cuda.is_available():torch.cuda.empty_cache()
    status,output=run_reader_train(1)
if status:
    print('CTC追加学習が失敗しました。',flush=True)
    print('ログ:',LOGFILE,flush=True)
    if 'backbone incompatible' in output.lower():
        print('英語Reader初期重みの形式が一致しません。⑨で INITIALIZE_FROM_ENGLISH=False を試す前に、ログに出た重みの差を確認してください。')
    elif 'out of memory' in output.lower():
        print('T4メモリ不足です。BATCH_SIZE=1、短めの音声を使用してください。')
    elif 'oov' in output.lower() or 'vocab' in output.lower():
        print('語彙やトークン化の問題です。⑥の語彙生成と⑩のOOV件数を確認してください。')
    raise RuntimeError(f'⑩ Reader学習失敗 (exit={status})。上に表示された EG2 Reader CTC training failed のtracebackを確認してください。')
print('学習済み Reader:',TRAINED/'reader_ja.pt')


⑩でエラーが出たら `!tail -n 120 /content/eg2_mix/eg2_train_ctc.log` を実行してください。
また、⑧が成功済みなら新しいNotebookで**③（スクリプト展開）と⑩（学習）だけ**を更新して再実行できます。③は音声キャッシュを消しません。
英語Readerの初期化に失敗した場合はモデル構造の不一致がないかを表示します。初期化を無効にすると学習条件が大幅に変わるため、勝手にOFFにはしません。


In [ ]:
# @title ⑪ ネイティブC用 Reader／INT8辞書に変換して検査
import shutil,json,struct
subprocess.run([sys.executable,str(TOOLS/'convert_eg2_reader.py'),
    str(TRAINED/'reader_ja.pt'),str(READY/'reader.eg2r')],check=True)
subprocess.run([sys.executable,str(TOOLS/'convert_eg2_table.py'),
    str(TABLE),str(READY/'word_table.eg2q')],check=True)
shutil.copyfile(VOCAB,READY/'reader_vocab.txt')
with (READY/'word_table.eg2q').open('rb') as f:
    magic,version,nwords,width=struct.unpack('<4sIII',f.read(16))
assert magic==b'EG2Q' and version==1 and width==768
assert nwords==len(VOCAB.read_text(encoding='utf8').splitlines())
assert (READY/'word_table.eg2q').stat().st_size==16+nwords*768
shutil.copyfile(TRAINED/'training_info.json',READY/'training_info.json')
print('native bilingual Reader:',nwords,'tokens')

In [ ]:
# @title ⑫ 学習済み共通Reader + 日英語彙を mix.eg2m にまとめる
import zipfile, shutil
subprocess.run([sys.executable,str(TOOLS/'eg2_bundle.py'),
    '--mix-dir',str(READY),'--out',str(READY),'--force'],check=True)
with zipfile.ZipFile(OUTPUT,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
    z.write(READY/'mix.eg2m','models/mix.eg2m')
print('完成:',OUTPUT, 'MiB:',round(OUTPUT.stat().st_size/2**20,2))
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    dest=Path('/content/drive/MyDrive/EmbeddingGemma2');dest.mkdir(parents=True,exist_ok=True)
    shutil.copy2(OUTPUT,dest/OUTPUT.name)
if BROWSER_DOWNLOAD:
    from google.colab import files
    files.download(str(OUTPUT))

## C 側で日本語・英語を同時認識

英語版 Colab で作成した `models/encoder.eg2m` を使い、
このノートで作成した `models/mix.eg2m` を同じディレクトリに配置します。

```bash
make
./cats -a eg2-native-asr -m models -L auto -w mixed.wav
```

`-L auto` は自動で Reader を切り替えるのではなく、**日英共通 Reader 1つ**でCTC復号します。
`en.eg2m` + `ja.eg2m` の2ファイルを置いただけでは混合音声にはなりません。
FLEURSだけでなく混合言語の録音で訓練し、3種類（日・英・混合）の評価を行ってください。

Colab のランタイムで T4 GPU を選ぶと CUDA で動作します。初期値 `DEVICE_PREFERENCE='auto'` と `EMBEDDING_DTYPE='auto'` は T4 では FP32 に、BF16対応GPUでは BF16 に切り替わります。